# Chapter 12.8: Neuroscience-Inspired Recommendation Systems

## Learning Objectives

By the end of this notebook, you will be able to:

1. Implement curiosity-driven exploration based on intrinsic motivation theory
2. Model cognitive load and decision fatigue in recommendation interfaces
3. Integrate cognitive biases (anchoring, decoy effects) into recommendation design
4. Apply memory and forgetting curve models for re-recommendation timing
5. Design recommendation systems that support habit formation
6. Build attention models that respect human cognitive constraints
7. Evaluate recommendations through the lens of cognitive science

## Prerequisites

- Understanding of exploration-exploitation trade-offs (Part 10)
- Familiarity with recommendation evaluation metrics (Part 7)
- Basic probability and statistics
- PyTorch proficiency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part12/chapter_12.8_neuroscience.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part12/chapter_12.8_neuroscience.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cpu')

print("All imports successful!")

## 1. Curiosity-Driven Exploration

In neuroscience, **curiosity** is an intrinsic motivation that drives organisms to
explore novel and uncertain states. We can apply this to recommendation:

### Intrinsic Reward via Prediction Error

$$r_{\text{intrinsic}} = \|f_\theta(s_t, a_t) - s_{t+1}\|^2$$

High prediction error = surprising outcome = interesting to explore.

### Information Gain as Curiosity

$$r_{\text{info}} = D_{\text{KL}}\left[p(\theta | s_{t+1}) \| p(\theta | s_t)\right]$$

Curiosity = how much we learn about the user from showing this item.

> **💡 Concept:** Curiosity-driven exploration naturally balances relevance and novelty.
> Instead of randomly exploring, the system actively seeks items that are *informative*
> — items that would teach us the most about user preferences.

In [ ]:
class CuriosityDrivenRecommender:
    """Recommender that uses prediction error as intrinsic motivation."""
    
    def __init__(self, n_items=200, embed_dim=16, n_users=100):
        self.n_items = n_items
        self.embed_dim = embed_dim
        
        # True item features (unknown to the model)
        self.item_features = np.random.randn(n_items, embed_dim) * 0.3
        
        # User preferences (unknown to the model)
        self.user_prefs = np.random.randn(n_users, embed_dim) * 0.3
        
        # Model's belief about item features (starts with high uncertainty)
        self.item_mean = np.zeros((n_items, embed_dim))
        self.item_var = np.ones((n_items, embed_dim))  # Uncertainty
        
        # Interaction counts per item
        self.item_counts = np.zeros(n_items)
    
    def get_relevance(self, user_id):
        """Estimated relevance (exploitation)."""
        return self.item_mean @ self.user_prefs[user_id]
    
    def get_curiosity(self, user_id):
        """Information gain curiosity (exploration)."""
        # Curiosity is proportional to uncertainty in the user's preference direction
        user_dir = self.user_prefs[user_id] / (np.linalg.norm(self.user_prefs[user_id]) + 1e-8)
        directional_var = np.sum(self.item_var * user_dir ** 2, axis=1)
        return directional_var
    
    def recommend(self, user_id, k=10, curiosity_weight=0.5):
        """Recommend items balancing relevance and curiosity."""
        relevance = self.get_relevance(user_id)
        curiosity = self.get_curiosity(user_id)
        
        # Normalize both signals
        if relevance.std() > 0:
            relevance = (relevance - relevance.mean()) / (relevance.std() + 1e-8)
        if curiosity.std() > 0:
            curiosity = (curiosity - curiosity.mean()) / (curiosity.std() + 1e-8)
        
        combined = (1 - curiosity_weight) * relevance + curiosity_weight * curiosity
        top_items = np.argsort(combined)[-k:][::-1]
        return top_items
    
    def observe(self, item_id, feedback):
        """Update belief about item based on feedback."""
        self.item_counts[item_id] += 1
        # Bayesian update (simplified)
        lr = 1.0 / (1.0 + self.item_counts[item_id])
        self.item_mean[item_id] += lr * (feedback - self.item_mean[item_id])
        self.item_var[item_id] *= (1 - lr * 0.5)  # Reduce uncertainty


def simulate_curiosity_exploration(n_steps=200, curiosity_weight=0.5):
    """Simulate a curiosity-driven recommendation session."""
    rec = CuriosityDrivenRecommender()
    user_id = 0
    rewards = []
    curiosity_scores = []
    coverage = []
    seen_items = set()
    
    for step in range(n_steps):
        items = rec.recommend(user_id, k=5, curiosity_weight=curiosity_weight)
        
        # Simulate user interaction (true relevance + noise)
        best_item = items[0]
        seen_items.add(best_item)
        true_reward = np.dot(rec.user_prefs[user_id], rec.item_features[best_item])
        noisy_feedback = rec.item_features[best_item] + np.random.randn(rec.embed_dim) * 0.1
        rec.observe(best_item, noisy_feedback)
        
        rewards.append(true_reward)
        curiosity_scores.append(rec.get_curiosity(user_id).mean())
        coverage.append(len(seen_items))
    
    return rewards, curiosity_scores, coverage

# Compare different curiosity weights
weights = [0.0, 0.3, 0.5, 0.7, 1.0]
all_results = {}
for w in weights:
    rewards, curiosity, coverage = simulate_curiosity_exploration(curiosity_weight=w)
    all_results[w] = {'rewards': rewards, 'curiosity': curiosity, 'coverage': coverage}
    print(f"Curiosity weight={w:.1f}: Avg reward={np.mean(rewards):.4f}, "
          f"Coverage={coverage[-1]} items")

In [ ]:
# Visualize curiosity exploration
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cumulative reward
for w in weights:
    cumrew = np.cumsum(all_results[w]['rewards'])
    axes[0].plot(cumrew, label=f'w={w:.1f}', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cumulative Reward')
axes[0].set_title('Curiosity vs Exploitation: Cumulative Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Item coverage
for w in weights:
    axes[1].plot(all_results[w]['coverage'], label=f'w={w:.1f}', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Unique Items Explored')
axes[1].set_title('Item Coverage')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Uncertainty reduction
for w in [0.0, 0.5, 1.0]:
    axes[2].plot(all_results[w]['curiosity'], label=f'w={w:.1f}', linewidth=2)
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Average Uncertainty')
axes[2].set_title('Uncertainty Reduction Over Time')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Memory and Forgetting Curves

Ebbinghaus's **forgetting curve** describes how memory fades over time:

$$R(t) = e^{-t/S}$$

where $R$ is retention, $t$ is time since last exposure, and $S$ is memory strength.

### Application to Recommendation

- **Re-recommendation timing**: When should we suggest an item again?
- **Spaced repetition**: Gradually increase intervals between re-recommendations
- **Memory-aware ranking**: Recently seen items have lower utility

> **🔑 Pro Tip:** Users don't need to be reminded of items they just saw. But items seen
> weeks ago might be worth re-recommending if the user liked them. Model this decay!

In [ ]:
class MemoryAwareRecommender:
    """Recommender that models user memory with forgetting curves."""
    
    def __init__(self, n_items=100, decay_rate=0.1, base_strength=1.0):
        self.n_items = n_items
        self.decay_rate = decay_rate
        self.base_strength = base_strength
        
        # Item relevance scores
        self.item_relevance = np.random.rand(n_items)
        
        # Memory state: (last_seen_time, memory_strength, n_exposures)
        self.memory = {}
        self.current_time = 0
    
    def get_retention(self, item_id):
        """Get current memory retention for an item."""
        if item_id not in self.memory:
            return 0.0  # Never seen
        last_time, strength, n_exp = self.memory[item_id]
        dt = self.current_time - last_time
        return np.exp(-dt / (strength + 1e-8))
    
    def recommend(self, k=10, novelty_bonus=0.5):
        """Recommend items considering memory state."""
        scores = np.zeros(self.n_items)
        for i in range(self.n_items):
            retention = self.get_retention(i)
            # High retention = user remembers = lower utility of re-showing
            # But medium retention = spaced repetition sweet spot
            if retention > 0.8:  # Too fresh in memory
                memory_utility = -0.5
            elif retention > 0.3:  # Good time for reinforcement
                memory_utility = 0.3
            elif retention > 0:  # Fading, worth reminding
                memory_utility = 0.5
            else:  # Never seen: novelty
                memory_utility = novelty_bonus
            
            scores[i] = self.item_relevance[i] + memory_utility
        
        return np.argsort(scores)[-k:][::-1]
    
    def observe(self, item_id, liked=True):
        """Update memory after showing an item."""
        if item_id in self.memory:
            _, old_strength, n_exp = self.memory[item_id]
            # Spaced repetition: each exposure strengthens memory
            new_strength = old_strength * 1.5 if liked else old_strength * 0.8
            self.memory[item_id] = (self.current_time, new_strength, n_exp + 1)
        else:
            self.memory[item_id] = (self.current_time, self.base_strength, 1)
    
    def advance_time(self, dt=1):
        self.current_time += dt

# Visualize forgetting curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard forgetting curve
t = np.linspace(0, 30, 100)
for strength in [1, 3, 5, 10]:
    retention = np.exp(-t / strength)
    axes[0].plot(t, retention, label=f'Strength={strength}', linewidth=2)
axes[0].set_xlabel('Time since last exposure')
axes[0].set_ylabel('Memory Retention')
axes[0].set_title('Ebbinghaus Forgetting Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Spaced repetition with strengthening
times = [0, 1, 3, 7, 15, 30]
strength = 1.0
retention_history = []
t_continuous = []

for i in range(len(times) - 1):
    t_range = np.linspace(times[i], times[i+1], 50)
    for t_val in t_range:
        r = np.exp(-(t_val - times[i]) / strength)
        retention_history.append(r)
        t_continuous.append(t_val)
    strength *= 1.5  # Strengthen with each review

axes[1].plot(t_continuous, retention_history, linewidth=2, color='purple')
for t_val in times[:-1]:
    axes[1].axvline(x=t_val, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Memory Retention')
axes[1].set_title('Spaced Repetition: Memory Strengthening')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Cognitive Load and Decision Fatigue

Humans have limited cognitive bandwidth. Too many choices lead to **decision fatigue**
and reduced satisfaction (the "paradox of choice").

$$\text{Satisfaction} = \text{Quality}(\text{choice}) - \text{CognitiveCost}(|\text{options}|)$$

$$\text{CognitiveCost}(n) = \alpha \cdot \log(n) + \beta \cdot n^{\gamma}$$

> **⚠️ Common Pitfall:** More options is not always better. A well-curated list of 5 items
> often leads to higher satisfaction than 50 choices. Recommendation should *reduce* cognitive
> load, not increase it.

In [ ]:
def simulate_choice_satisfaction(n_options_range, n_simulations=1000):
    """Simulate user satisfaction vs number of options."""
    results = {}
    
    for n_options in n_options_range:
        satisfactions = []
        for _ in range(n_simulations):
            # Generate item qualities
            qualities = np.random.beta(2, 5, n_options)
            
            # User selects the best perceived item (with noise)
            perceived = qualities + np.random.randn(n_options) * 0.1
            chosen_idx = np.argmax(perceived)
            chosen_quality = qualities[chosen_idx]
            
            # Cognitive cost of evaluating options
            cognitive_cost = 0.05 * np.log(n_options + 1) + 0.002 * n_options
            
            # Regret: awareness of unchosen good options
            best_quality = qualities.max()
            regret = 0.1 * (best_quality - chosen_quality) * np.log(n_options + 1)
            
            satisfaction = chosen_quality - cognitive_cost - regret
            satisfactions.append(satisfaction)
        
        results[n_options] = {
            'mean': np.mean(satisfactions),
            'std': np.std(satisfactions)
        }
    
    return results

n_options_range = [1, 2, 3, 5, 8, 10, 15, 20, 30, 50]
choice_results = simulate_choice_satisfaction(n_options_range)

fig, ax = plt.subplots(figsize=(8, 5))
means = [choice_results[n]['mean'] for n in n_options_range]
stds = [choice_results[n]['std'] for n in n_options_range]
ax.errorbar(n_options_range, means, yerr=stds, fmt='o-', linewidth=2, capsize=5)
ax.axvline(x=n_options_range[np.argmax(means)], color='red', linestyle='--',
           label=f'Optimal: {n_options_range[np.argmax(means)]} options')
ax.set_xlabel('Number of Options Presented')
ax.set_ylabel('User Satisfaction')
ax.set_title('Paradox of Choice: Satisfaction vs Number of Options')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal number of options: {n_options_range[np.argmax(means)]}")

## 4. Cognitive Biases in Recommendation

Human decision-making is subject to systematic biases:

- **Anchoring**: First item seen influences perception of subsequent items
- **Decoy effect**: Adding a dominated option changes preference between two others
- **Position bias**: Items at the top of a list get more attention
- **Framing**: How items are described affects choice

> **💡 Concept:** Understanding cognitive biases is important both for ethical recommendation
> (avoiding manipulation) and for building better systems (de-biasing evaluation data).

In [ ]:
def simulate_position_bias(n_items=20, n_users=1000, attention_decay=0.8):
    """Simulate position bias in recommendation lists."""
    # Items have random true quality
    true_quality = np.random.rand(n_items)
    
    click_counts = np.zeros(n_items)
    quality_when_clicked = np.zeros(n_items)
    
    for _ in range(n_users):
        # Random ordering
        order = np.random.permutation(n_items)
        
        for position, item in enumerate(order):
            # Attention decreases with position
            attention = attention_decay ** position
            # Click probability = attention * quality
            click_prob = attention * true_quality[item]
            if np.random.rand() < click_prob:
                click_counts[item] += 1
                quality_when_clicked[item] += true_quality[item]
    
    return true_quality, click_counts

def simulate_anchoring_effect(n_trials=1000):
    """Simulate anchoring bias: first item affects rating of others."""
    ratings_after_high = []
    ratings_after_low = []
    
    for _ in range(n_trials):
        target_quality = 0.5  # Medium quality item
        
        # Condition 1: Anchor with high-quality item first
        anchor_high = 0.9
        perceived_after_high = target_quality - 0.15 * (anchor_high - target_quality)
        perceived_after_high += np.random.randn() * 0.1
        ratings_after_high.append(perceived_after_high)
        
        # Condition 2: Anchor with low-quality item first
        anchor_low = 0.1
        perceived_after_low = target_quality - 0.15 * (anchor_low - target_quality)
        perceived_after_low += np.random.randn() * 0.1
        ratings_after_low.append(perceived_after_low)
    
    return ratings_after_high, ratings_after_low

# Simulate and visualize
ratings_high, ratings_low = simulate_anchoring_effect()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anchoring effect
axes[0].hist(ratings_high, bins=30, alpha=0.6, label='After high anchor', color='blue')
axes[0].hist(ratings_low, bins=30, alpha=0.6, label='After low anchor', color='orange')
axes[0].axvline(x=0.5, color='red', linestyle='--', label='True quality')
axes[0].set_xlabel('Perceived Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Anchoring Effect on Perceived Quality')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Position bias
true_q, clicks = simulate_position_bias()
axes[1].scatter(true_q, clicks, alpha=0.6)
z = np.polyfit(true_q, clicks, 1)
p = np.poly1d(z)
axes[1].plot(np.sort(true_q), p(np.sort(true_q)), 'r--', linewidth=2)
axes[1].set_xlabel('True Item Quality')
axes[1].set_ylabel('Click Count')
axes[1].set_title('Position Bias: Clicks vs True Quality')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Anchoring: Avg rating after high anchor = {np.mean(ratings_high):.3f}")
print(f"           Avg rating after low anchor  = {np.mean(ratings_low):.3f}")
print(f"           True quality = 0.500")

## 5. Habit Formation in Recommendation

Recommendations can support positive habit formation using the **habit loop**:

1. **Cue**: Context that triggers the habit (time, location, mood)
2. **Routine**: The recommended action/content
3. **Reward**: Positive outcome that reinforces the loop

$$P(\text{habit}_t) = 1 - e^{-\lambda \cdot \text{streak}_t}$$

where $\text{streak}_t$ is the number of consecutive times the behavior occurred.

In [ ]:
class HabitFormationModel:
    """Model habit formation through recommendation."""
    
    def __init__(self, n_habits=5, habit_strength_rate=0.1, decay_rate=0.05):
        self.n_habits = n_habits
        self.strength_rate = habit_strength_rate
        self.decay_rate = decay_rate
        
        # Habit states
        self.habit_strength = np.zeros(n_habits)
        self.streak = np.zeros(n_habits, dtype=int)
        self.total_completions = np.zeros(n_habits, dtype=int)
    
    def get_habit_probability(self, habit_id):
        """Probability that user follows through on habit."""
        return 1 - np.exp(-self.strength_rate * self.habit_strength[habit_id])
    
    def recommend_cue(self, context_time):
        """Recommend which habit to cue based on context."""
        # Morning habits (0-1), midday (1-2), evening (2-3), etc.
        best_habit = int(context_time * self.n_habits / 24) % self.n_habits
        return best_habit
    
    def step(self, habit_id, completed):
        """Update habit state after attempt."""
        if completed:
            self.streak[habit_id] += 1
            # Habit strengthens with streaks (compounding effect)
            self.habit_strength[habit_id] += 1.0 + 0.1 * self.streak[habit_id]
            self.total_completions[habit_id] += 1
        else:
            self.streak[habit_id] = 0
            self.habit_strength[habit_id] *= (1 - self.decay_rate)
    
    def daily_decay(self):
        """Apply daily strength decay."""
        self.habit_strength *= (1 - self.decay_rate * 0.5)

# Simulate habit formation over 60 days
n_days = 60
model = HabitFormationModel(n_habits=3)
history = {'day': [], 'habit': [], 'strength': [], 'probability': [], 'completed': []}

for day in range(n_days):
    for habit_id in range(3):
        prob = model.get_habit_probability(habit_id)
        # External motivation helps in early days
        external_boost = 0.3 if day < 21 else 0.1
        completed = np.random.rand() < (prob + external_boost)
        
        history['day'].append(day)
        history['habit'].append(habit_id)
        history['strength'].append(model.habit_strength[habit_id])
        history['probability'].append(prob)
        history['completed'].append(completed)
        
        model.step(habit_id, completed)
    model.daily_decay()

# Visualize habit formation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for h_id in range(3):
    mask = [i for i, h in enumerate(history['habit']) if h == h_id]
    days = [history['day'][i] for i in mask]
    probs = [history['probability'][i] for i in mask]
    strengths = [history['strength'][i] for i in mask]
    axes[0].plot(days, probs, label=f'Habit {h_id}', linewidth=2)
    axes[1].plot(days, strengths, label=f'Habit {h_id}', linewidth=2)

axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Completion Probability')
axes[0].set_title('Habit Formation: Completion Probability')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Day')
axes[1].set_ylabel('Habit Strength')
axes[1].set_title('Habit Strength Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🏋️ Exercise 1: Implement Curiosity with Neural Network Predictor

Replace the Bayesian uncertainty with a neural network forward model.
Use the prediction error of the forward model as the curiosity signal.

In [ ]:
# TODO: Implement neural curiosity module
class NeuralCuriosity(nn.Module):
    def __init__(self, state_dim=16, action_dim=16, hidden_dim=32):
        super().__init__()
        # TODO: Forward model that predicts next state
        # TODO: Curiosity = prediction error
        pass
    
    def get_curiosity(self, state, action):
        # TODO: Return prediction error as curiosity
        pass
    
    def update(self, state, action, next_state):
        # TODO: Train forward model
        pass

print("Exercise 1: Implement neural curiosity module")

## 🏋️ Exercise 2: Position Bias Correction

Implement a propensity-based position bias correction for learning from
click data. Use inverse propensity weighting (IPW).

In [ ]:
# TODO: Implement position bias correction
def estimate_position_propensity(click_data, n_positions=20):
    """Estimate position bias propensities from click data."""
    # TODO: Estimate P(click | position) using randomized data
    pass

def ipw_debiased_training(model, click_data, propensities):
    """Train model with inverse propensity weighted loss."""
    # TODO: Weight each click by 1/P(click | position)
    pass

print("Exercise 2: Implement position bias correction")

## 🏋️ Exercise 3: Adaptive Recommendation List Length

Build a system that dynamically adjusts the number of recommendations
shown based on the user's cognitive state (estimated from interaction patterns).

In [ ]:
# TODO: Implement adaptive list length
class AdaptiveListRecommender:
    def __init__(self):
        # TODO: Track user engagement patterns to estimate cognitive state
        pass
    
    def estimate_cognitive_load(self, user_id):
        # TODO: Estimate current cognitive load from:
        # - Time of day, session duration, items already viewed
        # - Decision speed (time between clicks)
        pass
    
    def recommend(self, user_id):
        # TODO: Adjust list length based on cognitive load
        # High load -> fewer, higher-confidence items
        # Low load -> more diverse options
        pass

print("Exercise 3: Implement adaptive list length recommender")

## Summary

In this notebook, we explored neuroscience-inspired recommendation:

1. **Curiosity-driven exploration**: Using prediction error and information gain as intrinsic motivation
2. **Memory models**: Forgetting curves and spaced repetition for re-recommendation timing
3. **Cognitive load**: The paradox of choice and optimal list length
4. **Cognitive biases**: Anchoring, position bias, and their impact on evaluation
5. **Habit formation**: Using recommendations to support positive behavior change

### Key Takeaways

- Curiosity-driven exploration outperforms random exploration in long-term reward
- Memory-aware recommendations avoid redundancy and find optimal re-showing times
- Fewer, better-curated options often beat large recommendation lists
- Position and anchoring biases contaminate interaction data and must be corrected
- Ethical recommendation design should leverage biases for good, not exploitation